In [1]:

!java -version


openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [2]:
!wget https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz


--2026-02-19 14:23:01--  https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
Resolving downloads.apache.org (downloads.apache.org)... 135.181.214.104, 88.99.208.237, 2a01:4f9:3a:2c57::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|135.181.214.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 730107476 (696M) [application/x-gzip]
Saving to: ‘hadoop-3.3.6.tar.gz’

hadoop-3.3.6.tar.gz 100%[===================>] 696.28M  22.8MB/s    in 32s     

2026-02-19 14:23:33 (21.6 MB/s) - ‘hadoop-3.3.6.tar.gz’ saved [730107476/730107476]



In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += ":/content/hadoop-3.3.6/bin:/content/hadoop-3.3.6/sbin"


In [4]:
!grep -n JAVA_HOME /content/hadoop-3.3.6/etc/hadoop/hadoop-env.sh


37:#  JAVA_HOME=/usr/java/testing hdfs dfs -ls
47:# Technically, the only required environment variable is JAVA_HOME.
54:# export JAVA_HOME=


In [5]:
!sed -i '/JAVA_HOME/d' /content/hadoop-3.3.6/etc/hadoop/hadoop-env.sh


In [6]:
!hadoop version


Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /content/hadoop-3.3.6/share/hadoop/common/hadoop-common-3.3.6.jar


In [7]:
%%writefile /content/hadoop-3.3.6/etc/hadoop/core-site.xml
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
</configuration>


Overwriting /content/hadoop-3.3.6/etc/hadoop/core-site.xml


In [8]:
%%writefile /content/hadoop-3.3.6/etc/hadoop/hdfs-site.xml
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
</configuration>


Overwriting /content/hadoop-3.3.6/etc/hadoop/hdfs-site.xml


In [9]:
!readlink -f $(which java)


/usr/lib/jvm/java-17-openjdk-amd64/bin/java


In [10]:
!echo -e "\nexport HDFS_NAMENODE_USER=root\nexport HDFS_DATANODE_USER=root\nexport HDFS_SECONDARYNAMENODE_USER=root" \
>> /content/hadoop-3.3.6/etc/hadoop/hadoop-env.sh


In [11]:
!hdfs namenode -format


2026-02-19 14:25:19,012 INFO namenode.NameNode: STARTUP_MSG: 
/************************************************************
STARTUP_MSG: Starting NameNode
STARTUP_MSG:   host = 9708b9e1e613/172.28.0.12
STARTUP_MSG:   args = [-format]
STARTUP_MSG:   version = 3.3.6
STARTUP_MSG:   classpath = /content/hadoop-3.3.6/etc/hadoop:/content/hadoop-3.3.6/share/hadoop/common/lib/netty-codec-stomp-4.1.89.Final.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/jetty-util-ajax-9.4.51.v20230217.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/netty-transport-native-kqueue-4.1.89.Final-osx-x86_64.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/stax2-api-4.2.1.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/gson-2.9.0.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/listenablefuture-9999.0-empty-to-avoid-conflict-with-guava.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/netty-transport-native-kqueue-4.1.89.Final-osx-aarch_64.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/paranamer-2.3.jar:/

In [12]:
!hdfs --daemon start namenode



In [13]:
!hdfs --daemon start datanode



In [14]:
!hdfs --daemon start secondarynamenode


In [15]:
!jps



1442 Jps
1267 NameNode
1335 DataNode
1406 SecondaryNameNode


In [16]:
!hdfs dfs -mkdir /ap


In [17]:
%%writefile mapper.py
#!/usr/bin/env python3
import sys

#loops reads each transaction one by one
for line in sys.stdin:
    items = line.strip().split(",")
    for item in items:
      item = item.strip()
      print(f"{item}\t1")


Writing mapper.py


In [18]:
%%writefile reducer.py
#!/usr/bin/env python3
import sys

current_item = None

current_count = 0
minimum_support = 2

for line in sys.stdin:
    item, count = line.strip().split("\t")
    count = int(count)

    if current_item == item:
        current_count += count
    else:
        if current_item and current_count >= minimum_support:
            print(f"{current_item}\t{current_count}")

        current_item = item
        current_count = count
if current_item and current_count >= minimum_support:
    print(f"{current_item}\t{current_count}")

Writing reducer.py


In [19]:
!chmod +x mapper.py reducer.py

In [ ]:
# give the transaction.csc file locally fron pc upload on this env

%%bash
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export HADOOP_HOME=/content/hadoop-3.3.6
export PATH=$PATH:$HADOOP_HOME/bin:$HADOOP_HOME/sbin

hdfs dfs -put -f transactions.csv /ap/

In [29]:
%%bash
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export HADOOP_HOME=/content/hadoop-3.3.6
export PATH=$PATH:$HADOOP_HOME/bin:$HADOOP_HOME/sbin

hdfs dfs -rm -r -f /ap/output

hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar \
-files mapper.py,reducer.py \
-input /ap/transactions.csv \
-output /ap/output \
-mapper "python3 mapper.py" \
-reducer "python3 reducer.py"


2026-02-19 14:49:46,800 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-02-19 14:49:47,017 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-02-19 14:49:47,017 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-02-19 14:49:47,043 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-02-19 14:49:47,558 INFO mapred.FileInputFormat: Total input files to process : 1
2026-02-19 14:49:47,701 INFO mapreduce.JobSubmitter: number of splits:1
2026-02-19 14:49:48,088 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local8728514_0001
2026-02-19 14:49:48,088 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-02-19 14:49:48,381 INFO mapred.LocalDistributedCacheManager: Localized file:/content/mapper.py as file:/tmp/hadoop-root/mapred/local/job_local8728514_0001_ef2340fe-9dc9-4a9f-af6f-bec3adddfa8a/mapper.py
2026-02-19 14:49:48,411 INFO mapred.LocalDistributedCacheManager:

In [31]:
!hdfs dfs -ls /ap


Found 2 items
drwxr-xr-x   - root supergroup          0 2026-02-19 14:49 /ap/output
-rw-r--r--   1 root supergroup        214 2026-02-19 14:40 /ap/transactions.csv


In [32]:
%%bash
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export HADOOP_HOME=/content/hadoop-3.3.6
export PATH=$PATH:$HADOOP_HOME/bin:$HADOOP_HOME/sbin

hdfs dfs -cat /ap/output/part-00000